# Segmentation alimentaire avec FoodSeg103 et YOLOv11-seg

## Objectif

Ce notebook entraîne un modèle de segmentation alimentaire à partir du dataset FoodSeg103.

L’objectif est de segmenter les zones alimentaires dans une image afin d’estimer la proportion occupée par les aliments dans un plat.

## Dataset

Source : FoodSeg103 sur Kaggle  
Chemin Kaggle : `/kaggle/input/datasets/fontainenathan/foodseg103`

Le dataset contient :
- des images dans `Images/img_dir`
- des masques de segmentation dans `Images/ann_dir`
- les listes d’entraînement et de test dans `ImageSets`

## Sorties

Le notebook génère :
- un dataset converti au format YOLO segmentation
- un modèle entraîné `best.pt`
- des résultats de validation

In [ ]:
# Installation de la bibliothèque Ultralytics pour utiliser YOLO.
!pip install ultralytics

In [ ]:
# Import des librairies nécessaires.
import os
import cv2
import shutil
import numpy as np
from pathlib import Path
from ultralytics import YOLO
from IPython.display import FileLink

In [ ]:
# Chemin racine du dataset FoodSeg103.
ROOT = "/kaggle/input/datasets/cleristonmartinelo/foodsegmentation"

# Dossiers des images et des masques.
IMG_DIR = f"{ROOT}/Images/img_dir"
ANN_DIR = f"{ROOT}/Images/ann_dir"

# Fichier des catégories.
CATEGORY_FILE = f"{ROOT}/category_id.txt"

# Listes train/test.
TRAIN_LIST = f"{ROOT}/ImageSets/train.txt"
TEST_LIST = f"{ROOT}/ImageSets/test.txt"

# Dossier de sortie converti au format YOLO.
DEST = "/kaggle/working/dataset"

print("ROOT:", ROOT)
print("IMG_DIR:", IMG_DIR)
print("ANN_DIR:", ANN_DIR)
print("CATEGORY_FILE:", CATEGORY_FILE)
print("TRAIN_LIST:", TRAIN_LIST)
print("TEST_LIST:", TEST_LIST)


In [ ]:
# Lecture des fichiers train.txt et test.txt.
# Les noms peuvent contenir déjà l'extension .jpg.
def lire_liste(path):
    with open(path, "r") as f:
        return [line.strip() for line in f if line.strip()]

train_files = lire_liste(TRAIN_LIST)
val_files = lire_liste(TEST_LIST)

print("Nombre train:", len(train_files))
print("Nombre val:", len(val_files))
print("Exemples train:", train_files[:5])
print("Exemples val:", val_files[:5])

In [ ]:
# Lecture des catégories FoodSeg103.
# Le masque utilise : 0 = background, 1 à 103 = classes alimentaires.
# YOLO doit utiliser : 0 à 102 = classes alimentaires, sans background.

categories_foodseg = {}

with open(CATEGORY_FILE, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        parts = line.split(None, 1)  # sépare l'ID et le nom, même si le nom contient des espaces
        if len(parts) == 2:
            id_foodseg = int(parts[0])
            nom_classe = parts[1].strip()
            categories_foodseg[id_foodseg] = nom_classe

# On ignore le background et on décale les IDs pour YOLO.
categories_yolo = {
    id_foodseg - 1: nom
    for id_foodseg, nom in categories_foodseg.items()
    if id_foodseg != 0
}

print("Nombre de classes YOLO:", len(categories_yolo))
print("Exemples:")
for i in list(categories_yolo.keys())[:10]:
    print(i, "->", categories_yolo[i])

# Exemple important : FoodSeg 66 = rice, donc YOLO 65 = rice.
print("FoodSeg 48 -> YOLO", 48 - 1, "=", categories_foodseg.get(48))
print("FoodSeg 66 -> YOLO", 66 - 1, "=", categories_foodseg.get(66))
print("FoodSeg 90 -> YOLO", 90 - 1, "=", categories_foodseg.get(90))


In [ ]:
# Création de la structure attendue par YOLO.
for d in ["images/train", "images/val", "labels/train", "labels/val"]:
    os.makedirs(f"{DEST}/{d}", exist_ok=True)

print("Structure YOLO créée.")

In [ ]:
# Conversion d'un masque PNG en fichier label YOLO segmentation multi-classes.
# Important : on conserve les vraies catégories du masque FoodSeg103.
# 0 = background, ignoré.
# 1 à 103 = classes alimentaires FoodSeg103.
# YOLO attend des classes de 0 à 102, donc : classe_yolo = valeur_masque - 1.

def mask_to_yolo_multiclass(mask_path, output_txt, min_area=20):
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    if mask is None:
        return False

    h, w = mask.shape
    lignes = []

    valeurs_classes = np.unique(mask)

    for valeur in valeurs_classes:
        if valeur == 0:
            continue  # background

        classe_yolo = int(valeur) - 1

        # Sécurité : ignorer une classe inconnue.
        if classe_yolo not in categories_yolo:
            continue

        masque_classe = (mask == valeur).astype(np.uint8) * 255

        contours, _ = cv2.findContours(
            masque_classe,
            cv2.RETR_EXTERNAL,
            cv2.CHAIN_APPROX_SIMPLE
        )

        for contour in contours:
            if len(contour) < 3:
                continue

            if cv2.contourArea(contour) < min_area:
                continue

            # Simplification légère pour éviter des polygones trop longs.
            epsilon = 0.001 * cv2.arcLength(contour, True)
            contour = cv2.approxPolyDP(contour, epsilon, True)

            if len(contour) < 3:
                continue

            points = contour.reshape(-1, 2)
            coords = []

            for x, y in points:
                coords.append(x / w)
                coords.append(y / h)

            ligne = str(classe_yolo) + " " + " ".join([f"{c:.6f}" for c in coords])
            lignes.append(ligne)

    with open(output_txt, "w", encoding="utf-8") as f:
        f.write("\n".join(lignes))

    return len(lignes) > 0


In [ ]:
# Fonction pour trouver le masque correspondant à une image.
# Exemple : 00000001.jpg -> 00000001.png

def trouver_masque(nom_image, split):
    base = Path(nom_image).stem
    mask_path = f"{ANN_DIR}/{split}/{base}.png"

    if os.path.exists(mask_path):
        return mask_path

    return None


# Conversion d'un split train ou test vers la structure YOLO.
def convertir_split(files, src_split, dst_split):
    ok = 0
    manquants = 0
    sans_masque_valide = 0

    for nom_image in files:
        img_path = f"{IMG_DIR}/{src_split}/{nom_image}"
        mask_path = trouver_masque(nom_image, src_split)

        if not os.path.exists(img_path) or mask_path is None:
            manquants += 1
            continue

        base = Path(nom_image).stem

        dst_img = f"{DEST}/images/{dst_split}/{nom_image}"
        dst_label = f"{DEST}/labels/{dst_split}/{base}.txt"

        shutil.copy(img_path, dst_img)

        if mask_to_yolo_multiclass(mask_path, dst_label):
            ok += 1
        else:
            sans_masque_valide += 1

    print(f"{dst_split} ok:", ok)
    print(f"{dst_split} manquants:", manquants)
    print(f"{dst_split} sans masque valide:", sans_masque_valide)


convertir_split(train_files, "train", "train")
convertir_split(val_files, "test", "val")


In [ ]:
# Vérification finale de la conversion.
print("Images train:", len(os.listdir(f"{DEST}/images/train")))
print("Labels train:", len(os.listdir(f"{DEST}/labels/train")))
print("Images val:", len(os.listdir(f"{DEST}/images/val")))
print("Labels val:", len(os.listdir(f"{DEST}/labels/val")))

print("Exemples labels:", os.listdir(f"{DEST}/labels/train")[:5])

In [ ]:
# Vérification du contenu d'un fichier label.
exemple_label = os.listdir(f"{DEST}/labels/train")[0]

with open(f"{DEST}/labels/train/{exemple_label}", "r") as f:
    print(f.read()[:500])

In [ ]:
# Création du fichier de configuration YOLO multi-classes.
# On utilise les 103 classes alimentaires de FoodSeg103, sans le background.

yaml_content = f"""
path: {DEST}
train: images/train
val: images/val

names:
"""

for idx in sorted(categories_yolo.keys()):
    nom = categories_yolo[idx].replace("'", "")
    yaml_content += f"  {idx}: '{nom}'\n"

with open("/kaggle/working/dataset.yaml", "w", encoding="utf-8") as f:
    f.write(yaml_content)

print(yaml_content[:1500])


In [ ]:
# Chargement du modèle YOLO segmentation pré-entraîné.
# Pour tester plus vite, tu peux garder yolo11n-seg.pt.
# Pour un meilleur résultat final, tu peux utiliser yolo11m-seg.pt.

model = YOLO("yolo11n-seg.pt")

# Entraînement du modèle multi-classes.
model.train(
    data="/kaggle/working/dataset.yaml",
    task="segment",
    epochs=50,
    imgsz=640,
    batch=8
)


In [ ]:
# Évaluation du modèle sur les données de validation.
metrics = model.val()
print(metrics)

In [ ]:
# Test sur des images de validation.
model.predict(
    source=f"{DEST}/images/val",
    conf=0.25,
    save=True,
    max_det=10
)

In [ ]:
# Trouver le modèle entraîné.
!find /kaggle/working -name "best.pt"

In [ ]:
# Copier best.pt dans un emplacement simple.
!cp /kaggle/working/runs/segment/train*/weights/best.pt /kaggle/working/best.pt

In [ ]:
# Créer un lien de téléchargement.
FileLink("/kaggle/working/best.pt")